In [1]:
# model.py
import numpy as np
import matplotlib.pyplot as plt

def bereken_ruimtepuzzel(
    neerslag_mm,
    afvoeren,
    flex_boven,
    flex_onder,
    vloerpeil,
    hoogte_weg,
    hoogte_groen,
    hoogte_droge_berging,
    max_peilstijging,
    peilstijging_droge_berging,
    opp_ha,
    open_water_pct,
    verhard_pct,
    droge_berging_pct,
    pct_natuurvriendelijk,
    talud_nvo,
    talud_normaal,
    beheer_vanaf_kant_pct,
    breedte_onderhoud,
    breedte_waterloop,
):
    # 1. Basisoppervlakken
    opp_m2 = opp_ha * 10_000
    open_water = open_water_pct * opp_m2
    verhard = verhard_pct * opp_m2
    droge_berging = droge_berging_pct * opp_m2
    onverhard = opp_m2 - open_water - verhard  # kan je nog verfijnen

    # 2. Berging in oppervlaktewater (vereenvoudigd)
    neerslag_m = neerslag_mm / 1000
    berging_oppervlaktewater = open_water * neerslag_m  # m3

    # 3. Taluds (normaal + natuurvriendelijk) – sterk geschematiseerd
    # hier kun je later exact de Excel-logica inbouwen
    breedte_profiel = breedte_waterloop
    diepte_bandbreedte = flex_boven - flex_onder  # m (negatieve NAP-waarden → let op teken)
    diepte_bandbreedte = abs(diepte_bandbreedte)

    # aanname: taludlengte = diepte * helling
    taludlengte_normaal = diepte_bandbreedte * talud_normaal
    taludlengte_nvo = diepte_bandbreedte * talud_nvo

    # oppervlak talud (per meter lengte watergang)
    talud_oppervlak_normaal = taludlengte_normaal  # m2 per m
    talud_oppervlak_nvo = taludlengte_nvo          # m2 per m

    # verhouding natuurvriendelijk
    frac_nvo = pct_natuurvriendelijk
    frac_normaal = 1 - frac_nvo

    talud_oppervlak_totaal = (
        talud_oppervlak_normaal * frac_normaal
        + talud_oppervlak_nvo * frac_nvo
    )

    # berging in taluds (vereenvoudigd: talud-oppervlak * peilstijging)
    peilstijging_m = max_peilstijging  # hier ga je nog koppelen aan scenario
    berging_taluds = talud_oppervlak_totaal * peilstijging_m * 1  # per meter lengte

    # 4. Bodemberging / droge berging (sterk geschematiseerd)
    # hier kun je poriënvolume, diepte etc. uit Excel overnemen
    porienvolume = 0.28  # placeholder
    bodemberging_m = porienvolume * 0.5  # bijv. 0.5 m effectieve diepte
    berging_onverhard = onverhard * bodemberging_m

    # 5. Ruimtebeslag (voor grafiek)
    ruimte = {
        "open_water_m2": open_water,
        "verhard_m2": verhard,
        "droge_berging_m2": droge_berging,
        "onverhard_overig_m2": max(onverhard - droge_berging, 0),
    }

    # 6. Resultaten bundelen
    resultaten = {
        "berging_oppervlaktewater_m3": berging_oppervlaktewater,
        "berging_taluds_m3_per_m": berging_taluds,
        "berging_onverhard_m3": berging_onverhard,
        "ruimte": ruimte,
        "profiel_params": {
            "flex_boven": flex_boven,
            "flex_onder": flex_onder,
            "vloerpeil": vloerpeil,
            "hoogte_weg": hoogte_weg,
            "hoogte_groen": hoogte_groen,
            "hoogte_droge_berging": hoogte_droge_berging,
            "breedte_waterloop": breedte_waterloop,
            "breedte_onderhoud": breedte_onderhoud,
        },
    }

    return resultaten


def plot_profiel(profiel_params):
    """
    Maakt een dwarsprofiel-plot:
    - watergang met bandbreedte
    - taluds (schematisch)
    - maaiveld/groen
    - as-weg
    - vloerpeil
    """
    flex_boven = profiel_params["flex_boven"]
    flex_onder = profiel_params["flex_onder"]
    vloerpeil = profiel_params["vloerpeil"]
    hoogte_weg = profiel_params["hoogte_weg"]
    hoogte_groen = profiel_params["hoogte_groen"]
    hoogte_droge_berging = profiel_params["hoogte_droge_berging"]
    b_water = profiel_params["breedte_waterloop"]
    b_onderhoud = profiel_params["breedte_onderhoud"]

    fig, ax = plt.subplots(figsize=(8, 4))

    # x-as: simpel profiel
    x = [0, b_water]
    y_boven = [flex_boven, flex_boven]
    y_onder = [flex_onder, flex_onder]

    # watergang
    ax.fill_between(x, y_onder, y_boven, color="#a6cee3", alpha=0.7, label="Watergang bandbreedte")

    # maaiveld/groen (schematisch links en rechts)
    ax.hlines(hoogte_groen, -b_onderhoud, 0, colors="green", linewidth=3, label="Groen")
    ax.hlines(hoogte_groen, b_water, b_water + b_onderhoud, colors="green", linewidth=3)

    # as-weg (bijv. rechts)
    ax.hlines(hoogte_weg, b_water + b_onderhoud, b_water + b_onderhoud + 5, colors="gray", linewidth=4, label="As weg")

    # vloerpeil (bijv. ergens rechts)
    ax.hlines(vloerpeil, b_water + b_onderhoud + 5, b_water + b_onderhoud + 10, colors="orange", linewidth=3, label="Vloerpeil")

    # droge berging (optioneel)
    if hoogte_droge_berging is not None:
        ax.hlines(hoogte_droge_berging, -b_onderhoud - 5, -b_onderhoud, colors="brown", linewidth=3, label="Droge berging")

    ax.invert_yaxis()  # omdat NAP vaak negatief is
    ax.set_xlabel("Dwarsrichting (m)")
    ax.set_ylabel("Hoogte (m NAP)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    return fig


def plot_ruimtebeslag(ruimte_dict):
    labels = list(ruimte_dict.keys())
    waarden = [v / 10_000 for v in ruimte_dict.values()]  # m2 → ha

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(labels, waarden, color=["#1f78b4", "#b2df8a", "#fb9a99", "#cab2d6"])
    ax.set_ylabel("Oppervlak (ha)")
    ax.set_title("Ruimtebeslag")
    plt.xticks(rotation=20, ha="right")

    return fig

In [3]:
# app.py
import streamlit as st
import model

st.set_page_config(page_title="Hoogte- en Ruimtepuzzel", layout="wide")

st.title("Hoogte- en Ruimtepuzzel Bleizo")

# --- 1. Neerslagscenario & afvoer ---
st.header("Neerslagscenario en afvoer")

scenario_opties = {
    "KNMI'24 - 2100H (48 uur)": 142.6,
    "KNMI'24 - 2100H (24 uur)": 133.0,
    "Limburgbui (48 uur)": 200.0,
    "KNMI'24 - 2150H (48 uur)": 153.0,
}

scenario_naam = st.selectbox("Neerslagscenario", list(scenario_opties.keys()))
neerslag_mm = scenario_opties[scenario_naam]

afvoeren_ja_nee = st.selectbox("Afvoeren", ["nee", "ja"])
afvoeren = afvoeren_ja_nee == "ja"

st.write(f"**Gekozen scenario:** {scenario_naam} → {neerslag_mm} mm")

# --- 2. Inrichting / hoogtes ---
st.header("Inrichting en hoogtes")

col1, col2, col3 = st.columns(3)

with col1:
    flex_boven = st.number_input("Flexibel peil bovenkant (m NAP)", value=-5.8, step=0.1)
    flex_onder = st.number_input("Flexibel peil onderkant (m NAP)", value=-6.2, step=0.1)

with col2:
    vloerpeil = st.number_input("Vloerpeil (m NAP)", value=-4.5, step=0.1)
    hoogte_weg = st.number_input("Hoogte as weg (m NAP)", value=-5.3, step=0.1)
    hoogte_groen = st.number_input("Hoogte groen (m NAP)", value=-5.3, step=0.1)

with col3:
    hoogte_droge_berging = st.number_input("Hoogte droge berging (m NAP, leeg = geen)", value=-5.3, step=0.1)
    max_peilstijging = st.number_input("Maximale peilstijging (m)", value=0.8, step=0.05)
    peilstijging_droge_berging = st.number_input("Peilstijging droge berging (m)", value=0.0, step=0.05)

# eenvoudige validaties
if vloerpeil < flex_boven or hoogte_weg < flex_boven or hoogte_groen < flex_boven:
    st.warning("Let op: vloerpeil, weg en groen mogen niet lager zijn dan de bovenkant bandbreedte.")

# --- 3. Oppervlakken en percentages ---
st.header("Oppervlak en gebruik")

col4, col5 = st.columns(2)

with col4:
    opp_ha = st.number_input("Oppervlak plangebied (ha)", value=36.4, step=0.1)
    open_water_pct = st.number_input("Open water (%)", value=5.0, step=0.5) / 100
    verhard_pct = st.number_input("Verhard (%)", value=57.0, step=0.5) / 100
    droge_berging_pct = st.number_input("Droge berging (onverhard) (%)", value=15.0, step=0.5) / 100

with col5:
    pct_natuurvriendelijk = st.number_input("Percentage natuurvriendelijk talud (%)", value=50.0, step=5.0) / 100
    talud_nvo = st.number_input("Talud natuurvriendelijk (1:x)", value=3.0, step=0.5)
    talud_normaal = st.number_input("Talud normaal (1:x)", value=2.0, step=0.5)
    beheer_vanaf_kant_pct = st.number_input("Beheer vanaf kant (%)", value=100.0, step=5.0) / 100
    breedte_onderhoud = st.number_input("Breedte onderhoudsstrook (m)", value=5.0, step=0.5)
    breedte_waterloop = st.number_input("Aanname breedte waterloop (m)", value=8.0, step=0.5)

# --- 4. Berekeningen ---
if st.button("Bereken en visualiseer"):
    resultaten = model.bereken_ruimtepuzzel(
        neerslag_mm=neerslag_mm,
        afvoeren=afvoeren,
        flex_boven=flex_boven,
        flex_onder=flex_onder,
        vloerpeil=vloerpeil,
        hoogte_weg=hoogte_weg,
        hoogte_groen=hoogte_groen,
        hoogte_droge_berging=hoogte_droge_berging,
        max_peilstijging=max_peilstijging,
        peilstijging_droge_berging=peilstijging_droge_berging,
        opp_ha=opp_ha,
        open_water_pct=open_water_pct,
        verhard_pct=verhard_pct,
        droge_berging_pct=droge_berging_pct,
        pct_natuurvriendelijk=pct_natuurvriendelijk,
        talud_nvo=talud_nvo,
        talud_normaal=talud_normaal,
        beheer_vanaf_kant_pct=beheer_vanaf_kant_pct,
        breedte_onderhoud=breedte_onderhoud,
        breedte_waterloop=breedte_waterloop,
    )

    col_a, col_b = st.columns(2)

    with col_a:
        st.subheader("Dwarsprofiel")
        fig_profiel = model.plot_profiel(resultaten["profiel_params"])
        st.pyplot(fig_profiel)

    with col_b:
        st.subheader("Ruimtebeslag")
        fig_ruimte = model.plot_ruimtebeslag(resultaten["ruimte"])
        st.pyplot(fig_ruimte)

    st.subheader("Kerncijfers")
    st.write({
        "Berging oppervlaktewater (m3)": round(resultaten["berging_oppervlaktewater_m3"], 1),
        "Berging taluds (m3 per m)": round(resultaten["berging_taluds_m3_per_m"], 1),
        "Berging onverhard (m3)": round(resultaten["berging_onverhard_m3"], 1),
    })

ModuleNotFoundError: No module named 'model'

SyntaxError: invalid syntax (507122745.py, line 1)